- MSSV: 25410291
- Họ Tên: Đinh Xuân Sâm
- GDrive: [IE313 > notebooks](https://drive.google.com/drive/folders/1nap2ahjmAj6aj1cm6xr5nZDrBFeBjc00)
- Github: [sam-uit/IE313](https://github.com/sam-uit/IE313)


# Buổi 08 - Bài 07 - Đánh Giá Mô Hình

Mục tiêu:

- Hiểu đánh giá mô hình.
- Phân tích Over-fitting, Under-fitting và hiểu cách chọn mô hình.
- Hiểu hồi quy Ridge.

> Thang đo BLOOM.

Nội dung:

1. Kỹ thuật đánh giá mô hình.
2. Overfitting - Underfitting - Chọn mô hình.
3. Hồi quy Ridge

Bộ dữ liệu:

- [Lý thuyết và Thực hành](https://raw.githubusercontent.com/datasethub/ds105/master/Model-Evaluation-and-Refinement.csv)

## 0 - Load Dataset

### Import libraries

In [16]:
# import numpy & panda
import numpy as np
import pandas as pd

In [17]:
# import matplotlib
import matplotlib.pyplot as plt

In [18]:
# import seaborn
import seaborn as sns

### Data file

In [19]:
dataset_theory_url = "https://raw.githubusercontent.com/datasethub/ds105/master/Model-Evaluation-and-Refinement.csv"
dataset_theory_file = "Model-Evaluation-and-Refinement.csv"

In [20]:
# download the dataset file
import requests

try:
    dataset_theory_response = requests.get(dataset_theory_url)

    with open(dataset_theory_file, "wb") as file:
        file.write(dataset_theory_response.content)

    print("Download complete!")
except:
    print("Download failed!")

Download complete!


### `read_csv`

In [21]:
# read from a local file
df_theory = pd.read_csv(dataset_theory_file)

In [22]:
df_theory

,symboling,normalized-losses,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,...,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,horsepower-binned,diesel,gas
0,1,161,mitsubishi,turbo,two,hatchback,fwd,front,93.0,0.755887,...,7.6,102.0,5500.0,24,30,7689,9.791667,Medium,0,1
1,-1,93,mercedes-benz,turbo,four,sedan,rwd,front,115.6,0.973570,...,21.5,123.0,4350.0,22,25,31600,10.681818,Medium,1,0
2,1,154,plymouth,std,four,hatchback,fwd,front,93.7,0.755887,...,9.4,68.0,5500.0,31,38,6229,7.580645,Low,0,1
3,-1,122,toyota,std,four,wagon,rwd,front,104.5,0.902451,...,9.2,156.0,5200.0,19,24,15750,12.368421,High,0,1
4,2,94,volkswagen,std,four,sedan,fwd,front,97.3,0.825084,...,23.0,52.0,4800.0,37,46,7995,6.351351,Low,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,-1,95,volvo,std,four,sedan,rwd,front,109.1,0.907256,...,8.8,134.0,5500.0,18,23,21485,13.055556,Medium,0,1
197,1,125,mitsubishi,turbo,four,sedan,fwd,front,96.3,0.828448,...,7.5,116.0,5500.0,23,30,9279,10.217391,Medium,0,1
198,3,150,mazda,std,two,hatchback,rwd,front,95.3,0.812110,...,9.4,101.0,6000.0,17,23,13645,13.823529,Low,0,1
199,3,122,alfa-romero,std,two,convertible,rwd,front,88.6,0.811148,...,9.0,111.0,5000.0,21,27,13495,11.190476,Medium,0,1


## 1 - Kỹ Thuật Đánh Giá Mô Hình

- Phương pháp đánh giá mô hình.
- Hiệu suất.
- Kiểm chứng chéo.
- Tóm tắt các thang đo đánh giá.

### Giới thiệu

- [The Data Science Methodology](https://medium.com/@ryo.kusnadi/the-data-science-methodology-6e87036e85d9)

### Training/Testing Set

Tách bộ dữ liệu thành:

- Training set (70%).
- Testing set (30%).
- Phát triển mô hình với tập training.
- Dùng testing để đánh giá hiệu suất dự đoán của mô hình.
- Khi hoàn thành đánh giá nên sử dụng alị tất cả dữ liệu để phát triển lại mô hình để có hiệu suất tốt nhất.

### Hàm `train_test_split()`

```python
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    test_size = 0.3,
    random_state = 0
)
```

- `x_data`: biến độc lập
- `y_data`: biến phụ thuộc (`df['price']`)
- `x_train`, `y_train`: biến độc lập và phụ thuộc của bộ `train`
- `x_test`, `y_test`: biến độc lập và phụ thuộc của bộ `test`
- `test_size`: kích cỡ của bộ `test`, đơn vị % (`0.3`, `0.4`, vv..)
- `random_state`: số dùng để làm cơ sở kiểm soát việc xáo trộn dữ liệu (seed?)

In [23]:
# lấy ra x_ và y_data
x1_data = df_theory[['highway-mpg']]
y1_data = df_theory['price']

In [24]:
# import thư viện
from sklearn.model_selection import train_test_split

In [25]:
# chia dữ liệu
train_test_split(
    x1_data,
    y1_data,
    test_size = 0.3,
    random_state = 0
)

[     highway-mpg
 96            37
 181           41
 19            46
 153           31
 92            38
 ..           ...
 67            25
 192           28
 117           39
 47            25
 172           33
 
 [140 rows x 1 columns],
      highway-mpg
 18            32
 170           25
 107           37
 98            33
 177           29
 ..           ...
 30            32
 160           25
 40            16
 56            47
 131           33
 
 [61 rows x 1 columns],
 96      6849
 181     5572
 19      7775
 153     7603
 92      6377
        ...  
 67     18399
 192    16845
 117    18344
 47     22625
 172     9095
 Name: price, Length: 140, dtype: int64,
 18      8499
 170    17710
 107     7198
 98      7775
 177     9980
        ...  
 30      6989
 160    17199
 40     45400
 56      7788
 131    10295
 Name: price, Length: 61, dtype: int64]

In [26]:
# khai báo các phần tử
x1_train, x1_test, y1_train, y1_test = train_test_split(
    x1_data,
    y1_data,
    test_size = 0.2,
    random_state = 0
)

In [27]:
# xem kích thước
x1_train.shape, x1_test.shape, y1_train.shape, y1_test.shape

((160, 1), (41, 1), (160,), (41,))

#### Ví dụ với m1

In [28]:
# import bảo bối
from sklearn.linear_model import LinearRegression
m1 = LinearRegression()

In [29]:
# xây dựng mô hình
m1.fit(x1_train, y1_train)

LinearRegression()

In [30]:
# tính mse và r2 cho train để quan sát
m1.score(x1_train, y1_train)

0.4963002789070682

In [32]:
# thử lại với tệp test
m1.score(x1_test, y1_test)

0.47262656964054517

Giá trị: 0.49xx cho kết quả không quá tốt, có thể cân nhắc đổi thành mô hình nhiều bậc hơn TRÊN CÙNG TỆP BIẾN (cùng đầu vào) và đánh giá lại với kỳ vọng kết quả tốt hơn và so sánh lại.

In [31]:
# dự đoán
y1train_hat = m1.predict(x1_train)

#### Ví dụ với m2 trên toàn bộ dữ liệu

In [33]:
# chọn biến
X2 = df_theory[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']]
y2 = df_theory['price']

# khai báo mô hình
m2 = LinearRegression()

# fit
m2.fit(X2, y2)

LinearRegression()

In [34]:
# dự đoán
y2hat = m2.predict(X2)

In [36]:
from sklearn.metrics import mean_squared_error

In [38]:
mean_squared_error(y2hat, y2)

11980366.87057861

In [39]:
# tính R2 cho m2
m2.score(X2, y2)

0.809356280660099

#### Ví dụ với m2 và train_test_split

Xây lại `m2` với bộ train/test

In [42]:
# lấy bộ dữ liệu cho m2
x2_data = df_theory[
    ['horsepower',
     'curb-weight',
     'engine-size',
     'highway-mpg']
]
y2_data = df_theory['price']

In [43]:
# training/test split
x2_train, x2_test, y2_train, y2_test = train_test_split(
    x2_data,
    y2_data,
    test_size = 0.2,
    random_state = 0
)

In [45]:
# xây dựng lại m2
m2 = LinearRegression()
m2.fit(x2_train, y2_train)

LinearRegression()

In [46]:
# đánh giá r2 của m2 với bộ train
m2.score(x2_train, y2_train)

0.8010488709456116

In [47]:
# đánh giá r2 của m2 với bộ test
m2.score(x2_test, y2_test)

0.8481242218902798

- 0.8xx, có nghĩa `m2` đã tốt hơn `m1`, hay nói cách khác, *phù hợp* hơn (với các biến này).
- Hay `m2` đang làm hài lòng cả tập tham gia làm công thức (training) và tập kiểm thử (test), có nghĩa `m2` *phù hợp* hơn.

#### Ví dụ với m3 - Hồi Quy Đa Thức trên toàn bộ dữ liệu

In [74]:
X3 = df_theory[['highway-mpg']]
y3 = df_theory['price']

In [75]:
# import PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures

In [85]:
# khai báo biến bậc 2
pf3 = PolynomialFeatures(degree=8, include_bias=False)

In [86]:
# biến đổi X3
X3_pf = pf3.fit_transform(X3)

In [87]:
print("Thực tế: ", X3.shape)
print("Biến đổi: ", X3_pf.shape)

Thực tế:  (201, 1)
Biến đổi:  (201, 8)


In [89]:
# khai báo
m3 = LinearRegression()

In [91]:
# fit
m3.fit(X3_pf, y3)

LinearRegression()

In [92]:
# dự đoán
y3hat = m3.predict(X3_pf)

In [93]:
# tính r2 của m3
m3.score(X3_pf, y3)

0.6806016846115149

Các giá trị tương ứng bậc:

- 1: 0.4965911884339177
- 2: 0.6517936037026723
- 3: 0.6741946663906521
- 8: 0.6806016846115149

#### Ví dụ với m3 với train_test_plit

In [84]:
# split
x3_train, x3_test, y3_train, y3_test = train_test_split(
    X3,
    y3,
    test_size = 0.2,
    random_state = 0
)

In [97]:
# pf

x3_train_pf = pf3.fit_transform(x3_train)
x3_test_pf = pf3.fit_transform(x3_test)

In [98]:
m3 = LinearRegression()
m3.fit(x3_train_pf, y3_train)

LinearRegression()

In [100]:
# theo dõi r2 với tập train
m3.score(x3_train_pf, y3_train)

0.6738166548904485

In [101]:
# r2 với tập test
m3.score(x3_test_pf, y3_test)

0.7027733941173993

### Bài Tập

- Bài 1. Chọn ra 4 biến và xây dựng mô hình hồi quy đa thức bậc 3, tăng bậc lên 4, 5, 6 và quan sát $R^2$, với toàn bộ dữ liệu và `train_test_split()`.
- Bài 2. Sử dụng toàn bộ tập 10 biến ở bài 06 (p5), thay đổi bậc từ 1, 2, 3, vv.. và quan sát $R^2$, với toàn bộ dữ liệu và `train_test_split()`.

## 2 - Overfitting - Underfitting - Chọn Mô Hình

TODO: Buổi 09

## 3 - Hồi Quy Ridge

#